In [7]:
import math
import time

import gradio as gr
import pandas as pd
import plotly.express as px
from datasets import load_dataset, Audio
from sdoml_task1.config import DATA_DIR
from sdoml_task1.modeling.predict import ModelLoader

from sdoml_task1.modeling.load import load_features
from sdoml_task1.modeling.train import run_training
from sdoml_task1.modeling.optimization import OPTIMIZERS, SCHEDULERS, REGULARIZATIONS, ACTIVATIONS
from sdoml_task1.plots import confusion_matrix_figure, training_metrics_figure


In [8]:
X_train, y_train, X_test, y_test = load_features()
df = pd.read_csv(DATA_DIR / "audio_mnist_metadata.csv")


MAX_LAYERS = 10


def update_slider_visibility(n_layers):
    """Show only the first ``n_layers`` hidden-layer inputs."""
    updates = []
    for i in range(MAX_LAYERS):
        if i < n_layers:
            updates.append(gr.Number(visible=True))
        else:
            updates.append(gr.Number(visible=False))
    return updates


def gradio_train(epochs, lr_text, batch_size, n_layers, activation_function, regularization,
                 reg_param, optimizer_name, scheduler_name, limit_epoch, progress=gr.Progress(), *layer_sizes):
    """Read the values from the interface, train the model and return a summary."""

    def show_epoch(epoch, total_epochs, train_loss, val_loss, val_acc):
        progress(epoch / total_epochs, desc=f"Epoch {epoch}/{total_epochs} - val accuracy {val_acc:.3f}")

    try:
        lr = float(lr_text)
    except ValueError:
        raise gr.Error(f"The learning rate must be a number (e.g. 0.001, got '{lr_text}'.")
    if lr <= 0:
        raise gr.Error("The learning rate must be positive.")

    hidden_sizes = []
    for i in range(int(n_layers)):
        if layer_sizes[i] is None or layer_sizes[i] < 1:
            raise gr.Error(f"Hidden layer {i+1} must have at least 1 neuron.")
        hidden_sizes.append(layer_sizes[i])

    if regularization == "None":
        regularization = None

    if regularization == "dropout":
        if reg_param < 0 or reg_param >= 1:
            raise gr.Error("For dropout, the value is between 0 and 1.")

    progress(0, desc="Starting training...")

    metrics = run_training(X_train, y_train, X_test, y_test, hidden_sizes=hidden_sizes,
                           activation_function=activation_function, regularization=regularization,
                           reg_param=reg_param, optimizer_name=optimizer_name, scheduler_name=scheduler_name,
                           epochs=int(epochs), lr=lr, batch_size=int(batch_size), limit_epoch=int(limit_epoch), on_epoch_end=show_epoch)

    epochs_done = len(metrics["loss_val"])
    final_acc = metrics["accuracy_val"][-1]
    best_acc = max(metrics["accuracy_val"])

    summary = (
        f"**Modele trained:**   \n"
        f"**Final validation accuracy:** {final_acc:.4f}    \n"
        f"**Best validation accuracy:** {best_acc:.4f}    \n"
        f"**Final validation loss:** {metrics['loss_val'][-1]:.4f}"
    )

    metrics_figure = training_metrics_figure(metrics)
    cm_figure = confusion_matrix_figure(
        metrics["confusion_matrices"][-1], len(metrics["confusion_matrices"])
    )

    return summary, metrics_figure, cm_figure


Features loaded from /home/amets/Documents/MASTER/SDOML/project/SDOML-project1/data/processed/features.pkl


In [9]:
# Get unique values for each column for the dropdown filters
def get_filter_options():
    filters = {}
    for col in df.columns:
        # Get unique values, excluding NaN
        unique_vals = df[col].dropna().astype(str).unique().tolist()
        # Sort and add "All" option
        filters[col] = ["All"] + sorted(unique_vals)
    return filters

filter_options = get_filter_options()

def filter_data(digit, speaker_id, gender, accent, age, origin, split):
    """Filter the dataframe based on selected criteria"""
    filtered_df = df.copy()

    # Apply filters (only if not "All" is selected)
    if digit != "All":
        filtered_df = filtered_df[filtered_df["digit"].astype(str) == digit]
    if speaker_id != "All":
        filtered_df = filtered_df[filtered_df["speaker_id"].astype(str) == speaker_id]
    if gender != "All":
        filtered_df = filtered_df[filtered_df["gender"].astype(str) == gender]
    if accent != "All":
        filtered_df = filtered_df[filtered_df["accent"].astype(str) == accent]
    if age != "All":
        filtered_df = filtered_df[filtered_df["age"].astype(str) == age]
    if origin != "All":
        filtered_df = filtered_df[filtered_df["origin"].astype(str) == origin]
    if split != "All":
        filtered_df = filtered_df[filtered_df["split"].astype(str) == split]

    # Return both the table and a count
    count = len(filtered_df)
    return filtered_df, f"Showing {count} entries"

def reset_filters():
    """Reset all filters to 'All'"""
    return "All", "All", "All", "All", "All", "All", "All", df, f"Showing {len(df)} entries"

# Map display names to actual column names
column_mapping = {
    "Digit": "digit",
    "Speaker ID": "speaker_id",
    "Gender": "gender",
    "Accent": "accent",
    "Age": "age",
    "Origin": "origin",
    "Split": "split"
}

def update_plot(digit, speaker_id, gender, accent, age, origin, split, plot_column):
    """Generate interactive plot for selected column based on filtered data"""
    filtered_df = df.copy()


    if digit != "All":
        filtered_df = filtered_df[filtered_df["digit"].astype(str) == digit]
    if speaker_id != "All":
        filtered_df = filtered_df[filtered_df["speaker_id"].astype(str) == speaker_id]
    if gender != "All":
        filtered_df = filtered_df[filtered_df["gender"].astype(str) == gender]
    if accent != "All":
        filtered_df = filtered_df[filtered_df["accent"].astype(str) == accent]
    if age != "All":
        filtered_df = filtered_df[filtered_df["age"].astype(str) == age]
    if origin != "All":
        filtered_df = filtered_df[filtered_df["origin"].astype(str) == origin]
    if split != "All":
        filtered_df = filtered_df[filtered_df["split"].astype(str) == split]

    # Map the display name to actual column name
    actual_column = column_mapping[plot_column]

    # Create value counts
    counts = filtered_df[actual_column].value_counts().reset_index()
    counts.columns = [actual_column, "count"]
    counts = counts.sort_values("count", ascending=False)

    # Create interactive bar chart with better styling
    fig = px.bar(
        counts,
        x=actual_column,
        y="count",
        title=f"Distribution by {plot_column}",
        labels={"count": "Count", actual_column: plot_column},
        color="count",
        color_continuous_scale="Viridis"
    )

    fig.update_layout(
        template="plotly_white",
        hovermode="x unified",
        title_font_size=20,
        xaxis_title_font_size=14,
        yaxis_title_font_size=14,
        height=500,
        showlegend=False,
        margin=dict(l=50, r=50, t=80, b=50),
        coloraxis_showscale=False
    )

    fig.update_traces(
        hovertemplate="<b>%{x}</b><br>Count: %{y}<extra></extra>",
        marker_line_color="white",
        marker_line_width=1
    )

    return fig

### Remember to close the port with .close() or by restarting the kernel. If you don't, it sometimes struggles to re-run the code cause the port is busy.

In [10]:
def create_app():
    with gr.Blocks() as interface:
        gr.Markdown(r"""
            # Spoken Digit Classification
            ### Project description
            This project uses the [Audio MNIST Dataset](https://huggingface.co/datasets/gilkeyio/AudioMNIST) to train a model that predicts numbers from 0 to 9 based on audio input. The neural network is a multilayer perceptron with an input of 26 neurons, two hidden layers of 64 and 32 neurons, and a final output layer of 10 neurons. It uses the Adam optimizer and the cross-entropy loss function.
        """)

        with gr.Tab():
            # ========================================= DATA EXPLORATION TAB =========================================
            with gr.Tab("Data"):
                gr.Markdown("The dataset has 50 English recordings per digit (0-9) of 60 speakers. There are 60 participants in total, with 12 being women and 48 being men, all featuring a diverse range of accents and country of origin. Their ages vary from 22 to 61 years old. This is a great dataset to explore a simple audio classification problem: either the digit or the gender.")
                with gr.Group():
                    gr.Markdown("Below you can ispect the dataset. Choose the filters you want and plot the data.")

                    with gr.Accordion("Filters", open=False):
                        with gr.Row():
                            digit_filter = gr.Dropdown(
                                choices=filter_options["digit"],
                                value="All",
                                label="Digit"
                            )
                            speaker_id_filter = gr.Dropdown(
                                choices=filter_options["speaker_id"],
                                value="All",
                                label="Speaker ID"
                            )
                            gender_filter = gr.Dropdown(
                                choices=filter_options["gender"],
                                value="All",
                                label="Gender"
                            )

                        with gr.Row():
                            accent_filter = gr.Dropdown(
                                choices=filter_options["accent"],
                                value="All",
                                label="Accent"
                            )
                            age_filter = gr.Dropdown(
                                choices=filter_options["age"],
                                value="All",
                                label="Age"
                            )
                            origin_filter = gr.Dropdown(
                                choices=filter_options["origin"],
                                value="All",
                                label="Origin"
                            )

                        with gr.Row():
                            split_filter = gr.Dropdown(
                                choices=filter_options["split"],
                                value="All",
                                label="Split"
                            )

                        # Reset button
                        with gr.Row():
                            reset_btn = gr.Button("Reset Filters", variant="secondary")

                    with gr.Accordion("Data table", open=False):
                        # Data table output
                        data_table = gr.Dataframe(
                            value=df,
                            interactive=False,
                            label="Data"
                        )

                        # Result count
                        result_text = gr.Textbox(
                            value=f"Showing {len(df)} entries",
                            interactive=False,
                            label="Result Count"
                        )

                    # Create event listeners for each filter
                    filters = [digit_filter, speaker_id_filter, gender_filter, accent_filter, age_filter, origin_filter, split_filter]

                    with gr.Accordion("Plots", open=True):
                        plot_column = gr.Dropdown(
                            choices=["Digit", "Speaker ID", "Gender", "Accent", "Age", "Origin", "Split"],
                            value="Digit",
                            label="Select Column to Plot"
                        )

                        plots_output = gr.Plot()

                        # Update plot when filters change
                        for filter_elem in filters:
                            filter_elem.change(
                                fn=update_plot,
                                inputs=filters + [plot_column],
                                outputs=[plots_output]
                            )

                        # Update plot when dropdown changes
                        plot_column.change(
                            fn=update_plot,
                            inputs=filters + [plot_column],
                            outputs=[plots_output]
                        )

                    for filter_elem in filters:
                        filter_elem.change(
                            fn=filter_data,
                            inputs=filters,
                            outputs=[data_table, result_text]
                        )

                    reset_btn.click(
                        fn=reset_filters,
                        inputs=None,
                        outputs=filters + [data_table, result_text]
                    )

            # ======================================== TRAINING INTERFACE TAB ========================================
            with gr.Tab("Train"):
                gr.Markdown("#### Training Interface")

                gr.Markdown("Choose the settings, then click **Train**. Note that if the number of epoch is high, it may take lot of time")
                with gr.Row():
                    with gr.Column(scale=2):
                        with gr.Row():
                            epochs_input = gr.Number(value=10, minimum=1, precision=0, label="Epochs")
                            lr_input = gr.Textbox(value="0.001", label="Learning rate")
                            batch_input = gr.Number(value=64, minimum=1, precision=0, label="Batch size")

                        n_layers_slider = gr.Slider(minimum=1, maximum=MAX_LAYERS, value=1, step=1, label="Number of hidden layers")

                        with gr.Row():
                            layer_inputs = []
                            for i in range(MAX_LAYERS):
                                layer_input = gr.Number(value=64, minimum=1, precision=0, label=f"Layer {i+1} neurons", visible=(i == 0))
                                layer_inputs.append(layer_input)

                        activation = gr.Radio(ACTIVATIONS, value="relu", label="Activation function")
                        optimizer = gr.Radio(OPTIMIZERS, value="Adam", label="Optimizer")
                        scheduler = gr.Radio(SCHEDULERS, value="None", label="Learning rate scheduler")
                        regularization = gr.Radio(REGULARIZATIONS, value="None", label="Regularization")

                        with gr.Row():
                            reg_param_input = gr.Number(value=0.0, minimum=0, label="Regularization strength", info="L1 / weight decay / dropout probability")
                            patience_input = gr.Number(value=10, minimum=1, precision=0, label="Early stopping patience", info="Only early stopping")

                        train_button = gr.Button("Train", variant="primary")

                    with gr.Column(scale=1):
                        summary_output = gr.Markdown("*Results will appear here after training.*")
                        gr.Markdown("#### Training results")
                        loss_plot = gr.Plot(label="Training metrics")
                        cm_plot = gr.Plot(label="Confusion matrix (last epoch)")

                n_layers_slider.change(fn=update_slider_visibility, inputs=n_layers_slider, outputs=layer_inputs)

                train_inputs = [epochs_input, lr_input, batch_input, n_layers_slider, activation, regularization, reg_param_input,
                                optimizer, scheduler, patience_input, *layer_inputs]
                train_button.click(
                    fn=gradio_train,
                    inputs=train_inputs,
                    outputs=[summary_output, loss_plot, cm_plot],
                )

            # ========================================= MODEL EVALUATION TAB =========================================
            with gr.Tab("Model"):
                loader = ModelLoader()

                if not loader.is_trained:
                    gr.Warning(
                        "No trained model was found: the interface runs on random weights."
                    )
                    gr.Markdown(
                        "**Warning: the model has not been trained.** The network below is randomly "
                        "initialized, so its predictions are meaningless. Train a model from the Train "
                        "tab (or `notebooks/01_audio_training.ipynb`), save it with `save_run()` and "
                        "restart the app."
                    )

                def predict_digit(audio_path: str) -> dict[str, float]:
                    if not loader.is_trained:
                        raise gr.Error(
                            "No trained model is loaded: train one first, then restart the app."
                        )
                    probs = loader.make_prediction(audio_path)
                    return {str(digit): float(p) for digit, p in enumerate(probs)}

                gr.Interface(
                    fn=predict_digit,
                    inputs=gr.Audio(type="filepath", label="Upload a spoken digit"),
                    outputs=gr.Label(num_top_classes=10, label="Predicted digit"),
                )

            gr.Markdown(r"""
                *Created by: Amets Martiarena, Teva Philippe, Alvaro Crespo | Check the original repo [here](https://github.com/AmetsMarti/SDOML-project1)*.
            """)
        return interface

In [11]:
try:
    interface.close()
except:
    pass

# Set to true to publish the app
send_it = False

interface = create_app()
interface.launch(server_port=7861, share=send_it, show_error=True)

Closing server running on port: 7861
* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Epoch [1/10], Train Loss: 2.2177, Val Loss: 0.5688, Val Acc: 0.8137
Epoch [2/10], Train Loss: 0.3739, Val Loss: 0.3433, Val Acc: 0.8830
Epoch [3/10], Train Loss: 0.2701, Val Loss: 0.3036, Val Acc: 0.8987
Epoch [4/10], Train Loss: 0.2302, Val Loss: 0.2913, Val Acc: 0.9035
Epoch [5/10], Train Loss: 0.1906, Val Loss: 0.2939, Val Acc: 0.9057
Epoch [6/10], Train Loss: 0.1853, Val Loss: 0.3181, Val Acc: 0.9038
Epoch [7/10], Train Loss: 0.1701, Val Loss: 0.3065, Val Acc: 0.9052
Epoch [8/10], Train Loss: 0.1626, Val Loss: 0.3478, Val Acc: 0.8948
Epoch [9/10], Train Loss: 0.1607, Val Loss: 0.3240, Val Acc: 0.9065
Epoch [10/10], Train Loss: 0.1492, Val Loss: 0.3185, Val Acc: 0.9067
Epoch [1/10], Train Loss: 2.2177, Val Loss: 0.5688, Val Acc: 0.8137
Epoch [2/10], Train Loss: 0.3739, Val Loss: 0.3433, Val Acc: 0.8830
Epoch [3/10], Train Loss: 0.2701, Val Loss: 0.3036, Val Acc: 0.8987
Epoch [4/10], Train Loss: 0.2302, Val Loss: 0.2913, Val Acc: 0.9035
Epoch [5/10], Train Loss: 0.1906, Val Loss: 0.2